In [1]:
!apt-get update
!apt-get install -y postgresql postgresql-contrib libpq-dev python3-dev build-essential

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,611 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [6,68

In [2]:
!service postgresql start


 * Starting PostgreSQL 14 database server
   ...done.


In [3]:
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"


ALTER ROLE


In [4]:
!sudo -u postgres psql -c "CREATE DATABASE milestone1_users;"


CREATE DATABASE


In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname="milestone1_users",
    user="postgres",
    password="postgres",
    host="localhost"
)

print("✅ PostgreSQL Connected Successfully!")
conn.close()


✅ PostgreSQL Connected Successfully!


In [1]:
!pip install streamlit psycopg2-binary bcrypt pyjwt watchdog


In [4]:
!pip install pyngrok


In [5]:
%%writefile app.py
import streamlit as st
import psycopg2
import jwt
import datetime
import bcrypt
import os
import re
import time

# ==============================
# CONFIGURATION
# ==============================

SECRET_KEY = "dev_secret_key"
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

DB_NAME = "milestone1_users"
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"

# ==============================
# CUSTOM UI STYLING
# ==============================

st.set_page_config(page_title="Auth System", layout="centered")

st.markdown("""
<style>
#MainMenu {visibility: hidden;}
footer {visibility: hidden;}
header {visibility: hidden;}

.stApp {
    background-color: #0E1117;
}

h1 {
    color: #4F8BF9;
    text-align: center;
}

.stButton > button {
    width: 100%;
    border-radius: 10px;
    height: 45px;
    background-color: #4F8BF9;
    color: white;
    font-weight: bold;
    border: none;
}

.stButton > button:hover {
    background-color: #3b6ccf;
}

.card {
    background-color: #262730;
    padding: 20px;
    border-radius: 15px;
    box-shadow: 0px 4px 15px rgba(0,0,0,0.4);
}
</style>
""", unsafe_allow_html=True)

# ==============================
# DATABASE
# ==============================

def get_connection():
    return psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )

def create_table():
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id SERIAL PRIMARY KEY,
            username VARCHAR(100) UNIQUE NOT NULL,
            email VARCHAR(150) UNIQUE NOT NULL,
            password TEXT NOT NULL,
            security_question TEXT NOT NULL,
            security_answer TEXT NOT NULL
        );
    """)
    conn.commit()
    cur.close()
    conn.close()

create_table()

# ==============================
# JWT
# ==============================

def create_access_token(data: dict):
    to_encode = data.copy()
    expire = datetime.datetime.utcnow() + datetime.timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    to_encode.update({"exp": expire})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)

def verify_token(token):
    try:
        return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
    except:
        return None

# ==============================
# VALIDATION
# ==============================

def is_valid_email(email):
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return re.match(pattern, email)

def is_valid_password(password):
    return len(password) >= 8 and password.isalnum()

# ==============================
# SESSION INIT
# ==============================

if "jwt_token" not in st.session_state:
    st.session_state["jwt_token"] = None

if "page" not in st.session_state:
    st.session_state["page"] = "login"

# ==============================
# SIGNUP
# ==============================

def signup_page():
    col1, col2, col3 = st.columns([1,2,1])
    with col2:
        st.markdown('<div class="card">', unsafe_allow_html=True)
        st.title("Create Account")

        username = st.text_input("Username")
        email = st.text_input("Email")
        password = st.text_input("Password", type="password")
        confirm_password = st.text_input("Confirm Password", type="password")

        question = st.selectbox("Security Question", [
            "What is your pet name?",
            "What is your mother’s maiden name?",
            "What is your favorite teacher?"
        ])

        answer = st.text_input("Security Answer")

        if st.button("Sign Up"):
            errors = []

            if not username:
                errors.append("Username required")
            if not email or not is_valid_email(email):
                errors.append("Valid email required")
            if not password or not is_valid_password(password):
                errors.append("Password must be 8+ alphanumeric")
            if password != confirm_password:
                errors.append("Passwords do not match")
            if not answer:
                errors.append("Security answer required")

            if errors:
                for e in errors:
                    st.error(e)
            else:
                try:
                    conn = get_connection()
                    cur = conn.cursor()
                    hashed_pw = bcrypt.hashpw(password.encode(), bcrypt.gensalt()).decode()

                    cur.execute("""
                        INSERT INTO users (username,email,password,security_question,security_answer)
                        VALUES (%s,%s,%s,%s,%s)
                    """,(username,email,hashed_pw,question,answer.lower()))

                    conn.commit()
                    cur.close()
                    conn.close()

                    st.success("Account created successfully!")
                    st.session_state["page"] = "login"
                    time.sleep(1)
                    st.rerun()

                except:
                    st.error("Username or Email already exists")

        if st.button("Back to Login"):
            st.session_state["page"] = "login"
            st.rerun()

        st.markdown('</div>', unsafe_allow_html=True)

# ==============================
# LOGIN
# ==============================

def login_page():
    col1, col2, col3 = st.columns([1,2,1])
    with col2:
        st.markdown('<div class="card">', unsafe_allow_html=True)
        st.title("Login")

        email = st.text_input("Email")
        password = st.text_input("Password", type="password")

        if st.button("Login"):
            conn = get_connection()
            cur = conn.cursor()
            cur.execute("SELECT username,password FROM users WHERE email=%s",(email,))
            result = cur.fetchone()
            cur.close()
            conn.close()

            if result:
                username_db, hashed_pw = result
                if bcrypt.checkpw(password.encode(), hashed_pw.encode()):
                    token = create_access_token({"sub":email,"username":username_db})
                    st.session_state["jwt_token"] = token
                    st.success("Login successful")
                    time.sleep(1)
                    st.rerun()
                else:
                    st.error("Incorrect password")
            else:
                st.error("Email not found")

        colA, colB = st.columns(2)
        with colA:
            if st.button("Create Account"):
                st.session_state["page"] = "signup"
                st.rerun()
        with colB:
            if st.button("Forgot Password"):
                st.session_state["page"] = "forgot"
                st.rerun()

        st.markdown('</div>', unsafe_allow_html=True)

# ==============================
# FORGOT PASSWORD
# ==============================

def forgot_password_page():
    st.title("Forgot Password")

    email = st.text_input("Enter Registered Email")

    if st.button("Verify Email"):
        conn = get_connection()
        cur = conn.cursor()
        cur.execute("SELECT security_question,security_answer FROM users WHERE email=%s",(email,))
        result = cur.fetchone()
        cur.close()
        conn.close()

        if result:
            question, correct_answer = result
            st.session_state["reset_email"] = email
            st.session_state["correct_answer"] = correct_answer
            st.session_state["security_question"] = question
        else:
            st.error("Email not found")

    if "security_question" in st.session_state:
        st.info(st.session_state["security_question"])
        user_answer = st.text_input("Your Answer")

        if st.button("Validate Answer"):
            if user_answer.lower() == st.session_state["correct_answer"]:
                reset_token = create_access_token({
                    "sub": st.session_state["reset_email"],
                    "type": "password_reset"
                })
                st.session_state["reset_token"] = reset_token
                st.success("Answer verified. Set new password.")
            else:
                st.error("Incorrect answer")

    if "reset_token" in st.session_state:
        payload = verify_token(st.session_state["reset_token"])
        if payload and payload.get("type") == "password_reset":
            new_password = st.text_input("New Password", type="password")
            confirm_password = st.text_input("Confirm Password", type="password")

            if st.button("Update Password"):
                if new_password != confirm_password:
                    st.error("Passwords do not match")
                elif not is_valid_password(new_password):
                    st.error("Invalid password format")
                else:
                    hashed_pw = bcrypt.hashpw(new_password.encode(), bcrypt.gensalt()).decode()
                    conn = get_connection()
                    cur = conn.cursor()
                    cur.execute("UPDATE users SET password=%s WHERE email=%s",
                                (hashed_pw, payload["sub"]))
                    conn.commit()
                    cur.close()
                    conn.close()

                    st.success("Password updated successfully!")
                    st.session_state.clear()
                    st.session_state["page"] = "login"
                    time.sleep(1)
                    st.rerun()

# ==============================
# DASHBOARD
# ==============================

def dashboard_page():
    token = st.session_state["jwt_token"]
    payload = verify_token(token)

    if not payload:
        st.session_state["jwt_token"] = None
        st.session_state["page"] = "login"
        st.rerun()
        return

    st.title(f"Welcome, {payload.get('username')}")
    st.success("You are logged in securely!")

    if st.button("Logout"):
        st.session_state["jwt_token"] = None
        st.session_state["page"] = "login"
        st.rerun()

# ==============================
# ROUTER
# ==============================

if st.session_state.get("jwt_token"):
    dashboard_page()
else:
    page = st.session_state.get("page", "login")

    if page == "signup":
        signup_page()
    elif page == "forgot":
        forgot_password_page()
    else:
        login_page()


Overwriting app.py


In [ ]:
from pyngrok import ngrok
import subprocess
import os
import time
import socket
# --- Wait for Streamlit to Start ---
def wait_for_streamlit(port=8501, timeout=60):
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(1)
            result = sock.connect_ex(('localhost', port))
            if result == 0:
                sock.close()
                return True
            sock.close()
        except Exception:
            pass
        time.sleep(1)
    return False
# --- Ngrok Setup ---
print("\nTo access the app, you need an Ngrok Authtoken.")
print("Get it from: https://dashboard.ngrok.com/get-started/your-authtoken")
authtoken = input("Enter your Ngrok Authtoken: ").strip()
if authtoken:
    ngrok.set_auth_token(authtoken)

    # Kill any existing ngrok process
    os.system("pkill ngrok")
    os.system("pkill streamlit")

    # Run Streamlit in the background FIRST
    print("Starting Streamlit...")
    # Using Subprocess.Popen to run in background
    # Redirecting output to /dev/null to keep cell clean
    process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"], stdout=subprocess.PIPE,
    stderr=subprocess.PIPE)

    # Wait for it to be ready
    if wait_for_streamlit():
        print("Streamlit is active! Connecting Ngrok...")
        # Open a tunnel to the streamlit port 8501
        try:
            public_url = ngrok.connect(8501).public_url
            print(f"\n🚀 Streamlit App is running!")
            print(f"👉 Public URL: {public_url}")
            print("\n(Click the URL above to open the app)")

            # Keep main thread alive
            try:
                # Keep checking if process is alive
                while process.poll() is None:
                    time.sleep(1)
            except KeyboardInterrupt:
                print("Stopping...")
                ngrok.disconnect(public_url)
                process.terminate()
        except Exception as e:
            print(f"Ngrok connection failed: {e}")
            process.terminate()
    else:
        print("Error: Streamlit failed to start in time.")
        process.terminate()
else:
    print("Ngrok Authtoken is required to expose the app publicly.")


To access the app, you need an Ngrok Authtoken.
Get it from: https://dashboard.ngrok.com/get-started/your-authtoken
